# Silver Layer Setup

This notebook runs locally in VS Code by default. If you want to use Google Colab, add a separate mounting cell there, but this notebook no longer depends on `google.colab`.

In [ ]:
import sys
print("Python version:", sys.version)
print("Setup verified ✅")

Python version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Setup verified âœ…


'\n\n**Click the Run button (â–¶) on the left of the cell.**\n\n---\n\n**What you should see:**\n'

In [3]:
!pip install pyspark pyiceberg pandas

In [ ]:
# ============================================================
# FinPulse - Silver Layer: Transaction Data
# Purpose: Clean, validate and enrich Bronze transaction data
# Author: Amanjot Kaur
# ============================================================
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import IntegerType
import os
from datetime import datetime

print("Imports successful")

# Paths - override with environment variables when needed
BRONZE_PATH = os.environ.get(
    "BRONZE_TRANSACTIONS_PATH",
    r"C:\FinPulse Project\data\bronze\transactions"
)

SILVER_OUTPUT_PATH = os.environ.get(
    "SILVER_OUTPUT_PATH",
    r"C:\FinPulse Project\data\silver\transactions"
)

ICEBERG_WAREHOUSE = os.environ.get(
    "ICEBERG_WAREHOUSE",
    r"C:\FinPulse Project\data\silver\iceberg_warehouse"
)

PIPELINE_VERSION = "v1.0"
INGESTION_TIMESTAMP = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

spark = (
    SparkSession.builder
    .appName("FinPulse-Silver-Transactions")
    .master("local[*]")
    .config(
        "spark.sql.extensions",
        "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions",
    )
    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.local.type", "hadoop")
    .config("spark.sql.catalog.local.warehouse", ICEBERG_WAREHOUSE)
    .config("spark.driver.memory", "2g")
    .config("spark.executor.memory", "2g")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")
print("Spark Session created")
print(f"Spark version: {spark.version}")

In [ ]:
def load_bronze_transactions(bronze_path):
    """
    Loads raw transaction data from Bronze layer.
    Finds latest partition automatically- portable across dates.
    """
    print("="*60)
    print("LOADING BRONZE TRANSACTION DATA")
    print("="*60)

    #Find latest date partition automatically
    partitions = [d for d in os.listdir(bronze_path)
                  if d.startswith("date=")]
    latest_partition = sorted(partitions)[-1]
    file_path = os.path.join(bronze_path, latest_partition, "transactions_raw.csv")
    print(f"Loading from partition: {latest_partition}")

    df = spark.read \
        .option("header","true") \
        .option("inferSchema","true") \
        .csv(file_path)
    
    print(f"Records loaded: {df.count():,}")
    print(f"Columns:{df.columns}")

    return df
bronze_df = load_bronze_transactions(BRONZE_PATH)
bronze_df.printSchema()

In [ ]:
def apply_silver_transformations(df):
    """Applies all Silver layer transformations."""
    print("=" * 60)
    print("APPLYING SILVER TRANSFORMATIONS")
    print("=" * 60)

    # Step 1: Remove zero amount transactions
    initial_count = df.count()
    df = df.filter(F.col("amount") > 0)
    removed = initial_count - df.count()
    print(f"Step 1 - Removed {removed:,} zero amount transactions")

    # Step 2: Add balance discrepancy flag
    df = df.withColumn(
        "is_balance_discrepancy",
        F.when(
            (F.col("type").isin(["TRANSFER", "CASH_OUT"]))
            & (F.col("oldbalanceOrg") > 0)
            & (
                F.round(F.col("oldbalanceOrg") - F.col("amount"), 2)
                != F.round(F.col("newbalanceOrig"), 2)
            ),
            True,
        ).otherwise(False),
    )
    discrepancies = df.filter(F.col("is_balance_discrepancy")).count()
    print(f"Step 2 - Balance discrepancies flagged: {discrepancies:,}")

    # Step 3: Convert step to simulation time fields
    df = df.withColumn("hour_of_simulation", F.col("step").cast(IntegerType()))
    df = df.withColumn("day_of_simulation", (F.col("step") / 24).cast(IntegerType()) + 1)
    print("Step 3 - Added hour_of_simulation and day_of_simulation")

    # Step 4: Encode transaction type
    df = df.withColumn(
        "type_encoded",
        F.when(F.col("type") == "TRANSFER", 0)
        .when(F.col("type") == "CASH_OUT", 1)
        .when(F.col("type") == "PAYMENT", 2)
        .when(F.col("type") == "DEBIT", 3)
        .when(F.col("type") == "CASH_IN", 4)
        .otherwise(-1),
    )
    print("Step 4 - Transaction type encoded")

    # Step 5: Add risk flag
    mean_amount = df.agg(F.mean("amount")).collect()[0][0]
    df = df.withColumn(
        "risk_flag",
        F.when(
            (F.col("type").isin(["TRANSFER", "CASH_OUT"]))
            & (F.col("amount") > mean_amount),
            True,
        ).otherwise(False),
    )
    risk_flagged = df.filter(F.col("risk_flag")).count()
    print(f"Step 5 - Risk flagged transactions: {risk_flagged:,}")

    # Step 6: Add balance difference column
    df = df.withColumn(
        "balance_diff",
        F.col("oldbalanceOrg") - F.col("newbalanceOrig"),
    )
    print("Step 6 - Added balance_diff column")

    # Step 7: Deduplicate
    before_dedup = df.count()
    df = df.dropDuplicates(["nameOrig", "step", "amount", "type"])
    after_dedup = df.count()
    print(f"Step 7 - Deduplicated transactions: {before_dedup - after_dedup:,} duplicates")

    # Step 8: Add pipeline metadata
    df = df.withColumn("silver_processed_at", F.lit(INGESTION_TIMESTAMP))
    df = df.withColumn("pipeline_version", F.lit(PIPELINE_VERSION))
    print("Step 8 - Added pipeline metadata")

    print("\nSilver transformations complete")
    print(f"Final record count: {df.count():,}")
    print(f"Final columns: {len(df.columns)}")

    return df


silver_df = apply_silver_transformations(bronze_df)
silver_df.printSchema()

In [ ]:
def validate_silver_data(df):
    """
    Basic validation checks before writing to Silver layer.
    Catches issues before bad data reaches Gold layer.
    """
    print("=" *60)
    print("SILVER DATA VALIDATION")
    print("="*60)

    checks_passed = 0
    checks_failed = 0
    # Check 1: No zero amounts
    zero_amounts = df.filter(F.col("amount")<=0).count()
    if zero_amounts == 0:
        print(" Check 1 PASSED - No zero amount transactions")
        checks_passed +=1
    else:
        print(f" check 1 Failed - {zero_amounts} zero amounts found")
        checks_failed +=1
    # Check 2: isFraud only 0 or 1
    invalid_fraud = df.filter( ~F.col("isFraud").isin([0,1])).count()
    if invalid_fraud == 0:
        print(" Check 2 PASSED - isFraud values valid")
        checks_passed+=1

    #Check 3 - No null amounts
    null_amounts = df.filter(F.col("amount").isNull()).count()
    if null_amounts == 0:
        print(" Check 3 PASSED - No null amounts")
        checks_passed+=1
    else:
        print(f" Check 3 Failed - {null_amounts} null amounts found")
        checks_failed +=1
    
    # Check 4 - Balance discrepancy fraud correlation
    discrepancy_fraud_rate = df.filter(F.col("is_balance_discrepancy")==True).agg(F.mean("isFraud")).collect()[0][0]
    print(f" Check 4 INFO — Fraud rate in discrepancy records: {discrepancy_fraud_rate:.2%}")
    checks_passed+=1
    print(f"\nValidation complete — "
          f"{checks_passed} passed, {checks_failed} failed")
    
    if checks_failed > 0:
        raise Exception(f" {checks_failed} validation checks failed. "
                        f"Pipeline stopped.")
     
    return True

validate_silver_data(silver_df)






